# Mixture of Experts: Repairing Where a Monotone Credit Model Is Weak

*Companion notebook for the MRM **Tabular** series.*

In the previous article we saw that a gradient-boosted tree ensemble is secretly a **kernel** —
two rows are similar if they fall in the **same leaves** — and that clustering the data in that
kernel exposes the regions where the model is weak.

Here we take the next two steps, on a **credit default** problem, using **Modeva** end to end:

1. Fit an **interpretable, monotone** base model — a **CatBoost with depth-2 trees** whose
   response is constrained to move in the economically correct direction for every driver.
2. Treat that CatBoost as a **kernel** and use a **Nyström spectral** embedding to find the
   clusters it handles worst — a *weakness map*.
3. Characterize the weak region with **PSI**, then **repair** it with a **Mixture of Experts**
   (monotone CatBoost experts) and confirm the lift lands on the weak clusters.

The whole pipeline keeps the monotonicity guarantee — base model, kernel, and every expert.

## Setup

Modeva is licence-gated. Generate a free licence at [modeva.ai](https://modeva.ai), then
uncomment and run the cell below (needed on Colab / a fresh environment).

In [ ]:
# !pip install -q modeva
# from modeva.utils.authenticate import authenticate
# authenticate(auth_code="PASTE_YOUR_LICENCE_CODE_HERE")

In [ ]:
import numpy as np
import pandas as pd
from modeva import DataSet, TestSuite
from modeva.models import (MoCatBoostClassifier, MoFuseKernelClassifier,
                           ModelTuneGridSearch, MoMoEClassifier)

## 1. Load the data

We work from a local `credit_default.csv` — 10,000 applicants, a binary `default` target
(~21% positive), and ten risk drivers (income, debt-to-income, credit score, utilization, …).
Modeva's `load_csv` reads it into a `DataSet`; `set_target` then infers the **classification**
task from the target column.

In [ ]:
ds = DataSet()
ds.load_csv("credit_default.csv")
ds.set_target("default")
ds.set_task_type("Classification")   # binary 0/1 default flag
print("task    :", ds.task_type)
print("target  :", ds.target_feature_name)
print("features:", list(ds.feature_names))

## 2. Train / test split

Split 80/20 with Modeva's `set_random_split`. The kernel-ridge fit later runs on a bounded
**Nyström** support set, so it scales roughly *linearly in `n`* — no subsampling needed.

In [ ]:
ds.set_random_split(test_ratio=0.2, random_state=0)
print("train:", ds.train_x.shape, "| test:", ds.test_x.shape)

## 3. Preprocess

Min-max scale the numerical features through Modeva's pipeline so train and test share one
consistent transform. The target is a 0/1 label, so it is left untouched. Min-max scaling is
**monotone increasing**, so it preserves the direction of every constraint we set next.

In [ ]:
ds.scale_numerical(features=tuple(ds.feature_names), method="minmax")
ds.preprocess()

## 4. Monotonicity constraints

A credit model should be **monotone** in its drivers: raising debt-to-income or utilization can
only *increase* estimated default risk; a higher credit score or income can only *decrease* it.
We encode these business priors as per-feature constraints (`+1` increasing, `-1` decreasing,
`0` free) w.r.t. `P(default = 1)`:

| Feature | Constraint | Reasoning |
|---|---|---|
| `dti`, `utilization`, `delinquencies` | **+1** | more debt burden / revolving use / past misses → more risk |
| `amount`, `tenure` | **+1** | larger loan / longer term → more exposure |
| `score`, `income`, `emp_length`, `savings` | **−1** | stronger credit / capacity / stability → less risk |
| `employment` | **0** | binary flag, no clear economic sign — left free |

CatBoost wants `monotone_constraints` as a **positional string** in the model's feature order.
(A `list`/`dict` breaks sklearn's `clone` — used by both the grid search and the Mixture of
Experts — while a `tuple` is rejected by CatBoost's fit; the string form satisfies both.)

In [ ]:
DIRECTION = {
    "dti": +1, "utilization": +1, "delinquencies": +1, "amount": +1, "tenure": +1,
    "score": -1, "income": -1, "emp_length": -1, "savings": -1,
    "employment": 0,
}
# positional string in the DataSet's feature order
mono = "(" + ",".join(str(DIRECTION[f]) for f in ds.feature_names) + ")"
print("feature order :", list(ds.feature_names))
print("monotone_cons :", mono)

## 5. The interpretable base model — CatBoost, depth 2, monotone

Depth-2 trees keep the model **interpretable**: each tree is a single pairwise interaction, so
the ensemble is essentially a functional-ANOVA model of main effects and low-order interactions.
We tune the number of trees and the learning rate by **5-fold cross-validation** (ranked by AUC),
carrying the depth-2 and monotonicity settings fixed throughout.

In [ ]:
base = MoCatBoostClassifier(depth=2, monotone_constraints=mono, verbose=0)
hpo = ModelTuneGridSearch(dataset=ds, model=base)
cv_result = hpo.run(
    param_grid={"iterations": [200, 400, 600], "learning_rate": [0.03, 0.05, 0.1]},
    metric=("AUC", "ACC"),
    cv=5,
)
cv_result.table

In [ ]:
# rank by cross-validated AUC and keep the best configuration
best_idx = cv_result.table["AUC"].idxmax()
best_params = dict(cv_result.value["params"][best_idx])
best_params

Retrain on the full training set with the best hyperparameters — a **free** (unconstrained) twin
and the **monotone** model — and compare. Monotonicity is a real constraint, and on this data it
carries a price: the free model scores higher out of the box.

In [ ]:
cb_free = MoCatBoostClassifier(name="Free", depth=2, verbose=0, **best_params)
cb_free.fit(ds.train_x, ds.train_y.ravel())

cb = MoCatBoostClassifier(name="Monotone", depth=2,
                          monotone_constraints=mono, verbose=0, **best_params)
cb.fit(ds.train_x, ds.train_y.ravel())

ts_free, ts = TestSuite(ds, cb_free), TestSuite(ds, cb)
pd.concat({"free":     ts_free.diagnose_accuracy_table().table.loc[["test"]],
           "monotone": ts.diagnose_accuracy_table().table.loc[["test"]]})

## 6. Inherent interpretability — FANOVA

Because the trees are **depth 2**, the ensemble is a **functional-ANOVA** model: every tree is a
single pairwise split, so the model decomposes exactly into **main effects** and **low-order
interactions**. Modeva reads that structure straight out of the fitted CatBoost — *inherent*
interpretability, not a post-hoc surrogate.

That is also where the free model's extra accuracy comes from — and why you might not want it. Read
the **free** model's `utilization` effect: it **wiggles**, reversing direction again and again. Higher
utilization making default *less* likely in stretches is not a real signal; it is the model
contorting to fit noise — exactly what a credit analyst (or a regulator) would reject.

In [ ]:
ts_free.interpret_effects(features="utilization", dataset="test").plot()   # free: wiggles, reverses

The **monotone** model's same effect is clean and strictly increasing — a shape you can sign off on.
This is the trade the accuracy table priced: a little discrimination given up for an effect that is
defensible everywhere.

In [ ]:
ts.interpret_effects(features="utilization", dataset="test").plot()        # monotone: clean, increasing

Now read the monotone model as a whole. **Effect importance** (`interpret_ei`) splits it into main
effects vs interactions, and **feature importance** (`interpret_fi`) ranks the drivers — MoCharts bars.

In [ ]:
ts.interpret_ei(dataset="test").plot()     # main-effect + interaction importance (FANOVA)

In [ ]:
ts.interpret_fi(dataset="test").plot()     # permutation feature importance

Now the **shape** of the strongest drivers. `interpret_effects` returns each effect as a curve;
because we imposed monotonicity, these are guaranteed monotone — `score` bends default risk
**down**, `dti` bends it **up** — a property you can read straight off the plot.

In [ ]:
ts.interpret_effects(features="score", dataset="test").plot()   # monotone decreasing

In [ ]:
ts.interpret_effects(features="dti", dataset="test").plot()     # monotone increasing

And a **pairwise interaction** surface — the second-order term a depth-2 tree can express:

In [ ]:
ts.interpret_effects(features=("dti", "score"), dataset="test").plot()

## 7. CatBoost as a kernel

Two applicants who land in the **same leaves** across the boosting rounds are treated as similar;
that leaf co-membership matrix is a valid kernel `K`. In Modeva we obtain exactly this kernel with
a `MoFuseKernelClassifier` restricted to the **tree channel only** (`use_rbf=False`,
`use_spectral=False`) — no RBF, no learned spectral component, just the pure CatBoost leaf-kernel.
We pass `backend="catboost"` with the **same tuned, monotone** `gbdt_params`, so the kernel comes
from the very model we validated above. `solver="nystrom"` keeps the fit linear in `n`.

In [ ]:
gbdt_params = {**best_params, "depth": 2, "monotone_constraints": mono, "verbose": 0}
fk = MoFuseKernelClassifier(
    name="CatBoost-Kernel", backend="catboost",
    use_xgb=True, use_rbf=False, use_spectral=False,   # pure leaf co-membership kernel
    solver="nystrom", gbdt_params=gbdt_params, random_state=0,
)
fk.fit(ds.train_x, ds.train_y.ravel())
print("pure CatBoost leaf-kernel fitted")

## 8. Nyström spectral clustering → weakness map

`diagnose_weak_clusters` builds a degree-normalised **Nyström** spectral embedding of that kernel,
clusters the applicants in the embedding, and reports the model metric **per cluster** on train and
test. Clusters with a low AUC or a large train/test gap are where the model is weak.

In [ ]:
weak = fk.diagnose_weak_clusters(ds, n_clusters=5)
weak.table

The MoCharts bar chart makes the weakest clusters obvious — the shortest bars are the regions the
model handles worst:

In [ ]:
weak.plot()

And the explicit weakest-cluster ranking:

In [ ]:
weak.value["worst_clusters"]

## 9. What characterizes the weak region?

Knowing *that* a cluster is weak only helps if we know *where* it sits in feature space.
`diagnose_weak_clusters` returns the per-sample cluster labels, so we split *weak cluster vs the
rest* and feed it to Modeva's distribution-shift test — the same **PSI** (Population Stability
Index) `data_drift_test` the resilience diagnostic uses.

In [ ]:
labels = np.asarray(weak.value["labels_train"])
worst_id = int(weak.value["worst_clusters"].iloc[0]["cluster"])
in_weak = labels == worst_id
drift = ds.data_drift_test(
    dataset1="train", sample_idx1=np.where(in_weak)[0],
    dataset2="train", sample_idx2=np.where(~in_weak)[0],
    name1=f"weak cluster {worst_id}", name2="rest",
    distance_metric="PSI",
)
drift.table   # PSI per feature, largest first

In [ ]:
drift.plot("summary")

Overlay the **density** of the highest-PSI feature, weak cluster vs the rest — the concrete
distribution shift that marks the region the model handles worst, and the starting point for a
targeted fix.

In [ ]:
top_feature = drift.table.index[0]
print("Largest-PSI feature:", top_feature)
drift.plot(("density", top_feature))

## 10. Repair with a Mixture of Experts

Instead of one global model straining to fit every region, a **Mixture of Experts** trains several
specialists and lets a gate route each applicant to the experts that serve its region best. We use
Modeva's `MoMoEClassifier` with **monotone CatBoost depth-2 experts** — so every expert keeps the
same economic guarantees as the base model — and one expert per diagnosed cluster.

This is where the accuracy we paid for monotonicity comes back. The single monotone model scored
test AUC ≈ 0.81; the free model, ≈ 0.92. The monotone Mixture of Experts lands in between — much
closer to the black box — **without ever leaving the monotone, interpretable family.**

In [ ]:
moe = MoMoEClassifier(name="MoE-CatBoost-d2", n_clusters=5, expert="catboost",
                      depth=2, monotone_constraints=mono, verbose=0)
moe.fit(ds.train_x, ds.train_y.ravel())
ts_moe = TestSuite(ds, moe)
ts_moe.diagnose_accuracy_table().table

The Mixture of Experts is **still fully interpretable and monotone**. `interpret_moe_cluster_analysis`
reports each expert's region and performance, and `interpret_effects_moe_average` shows the
gate-averaged FANOVA effect — which keeps the same monotone shape as the base model. Repair does
not cost us the economic guarantees.

In [ ]:
ts_moe.interpret_moe_cluster_analysis(dataset="test").plot()   # per-expert region + AUC

In [ ]:
ts_moe.interpret_effects_moe_average(features="dti", dataset="test").plot()   # still monotone up

In [ ]:
ts_moe.interpret_effects_moe_average(features="score", dataset="test").plot()  # still monotone down

## 11. Did the repair land on the weak clusters?

A higher headline AUC is only convincing if the lift shows up **in the weak regions**. We re-use the
kernel's test-set cluster labels and compare per-cluster test AUC, base model vs Mixture of Experts.

In [ ]:
from sklearn.metrics import roc_auc_score

lab_te = np.asarray(weak.value["labels_test"])
y_te = ds.test_y.ravel()
p_base = cb.predict_proba(ds.test_x)[:, 1]
p_moe = moe.predict_proba(ds.test_x)[:, 1]

rows = []
for c in sorted(np.unique(lab_te)):
    m = lab_te == c
    if m.sum() < 10 or len(np.unique(y_te[m])) < 2:
        auc_b = auc_m = np.nan
    else:
        auc_b = roc_auc_score(y_te[m], p_base[m])
        auc_m = roc_auc_score(y_te[m], p_moe[m])
    rows.append({"cluster": int(c), "n_test": int(m.sum()),
                 "base_auc": auc_b, "moe_auc": auc_m, "delta": auc_m - auc_b})
compare = pd.DataFrame(rows).sort_values("base_auc").reset_index(drop=True)
compare

## Takeaways

- A **monotone, depth-2 CatBoost** is **inherently interpretable** — a FANOVA decomposition of
  main effects and pairwise interactions, with monotone effect shapes you can read off directly —
  *and* a **kernel machine**, whose leaves induce a similarity kernel over the applicants.
- Monotonicity is not free here: it trades a little discrimination for effects that are defensible
  everywhere (no wiggles). That price is the model refusing to fit noise.
- A **Nyström** spectral embedding of the model's kernel turns it into cheap, proximity-based
  clusters, and scoring **per cluster** produces a *weakness map* — where the model actually fails.
- A **Mixture of Experts** with monotone experts **repairs** those weak regions and **buys back most
  of the accuracy** monotonicity cost — the lift concentrates exactly on the clusters the kernel
  flagged as weak, while the gate-averaged FANOVA effects stay **monotone and interpretable**.